In [16]:
import warnings
warnings.filterwarnings("ignore")

In [17]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4070


In [18]:
TICKERS = [
    "AAPL","AXP","KO","BAC","CVX",
    "MCO","OXY","CB","KHC","GOOGL"
]

START_DATE = "2021-01-01"
END_DATE = "2026-06-01"

FORECAST_HORIZON = 120  # ~6 months

TOP_K = 5

In [19]:
import yfinance as yf
print("Downloading data...")

raw = yf.download(
    TICKERS,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True
)

[*********************100%***********************]  10 of 10 completed

In [20]:
import pandas as pd
from ta.trend import SMAIndicator, EMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import BollingerBands

def create_features(df):
    df = df.copy()

    # SMA
    df["sma20"] = SMAIndicator(df["Close"], window=20).sma_indicator()
    df["sma50"] = SMAIndicator(df["Close"], window=50).sma_indicator()

    # EMA
    df["ema20"] = EMAIndicator(df["Close"], window=20).ema_indicator()

    # RSI
    df["rsi"] = RSIIndicator(df["Close"], window=14).rsi()

    # MACD
    macd = MACD(df["Close"])
    df["macd"] = macd.macd()
    df["macd_signal"] = macd.macd_signal()

    # Bollinger Bands
    bb = BollingerBands(df["Close"])
    df["bb_high"] = bb.bollinger_hband()
    df["bb_low"] = bb.bollinger_lband()

    # Returns
    df["returns"] = df["Close"].pct_change()

    # Volatility
    df["volatility"] = df["returns"].rolling(20).std()

    df = df.dropna()

    return df

In [21]:
def directional_accuracy(y_true, y_pred):
    actual_direction = np.sign(np.diff(y_true))
    pred_direction = np.sign(np.diff(y_pred))

    correct = (actual_direction == pred_direction).sum()
    total = len(actual_direction)

    return correct / total

In [22]:
EXPORT_FORECAST_REPORT_PLOT = "./reports/forecast_report.pdf"

In [23]:
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from matplotlib.backends.backend_pdf import PdfPages

all_forecasts = []
plot_pdf = PdfPages(EXPORT_FORECAST_REPORT_PLOT)
for ticker in TICKERS:
    df = pd.DataFrame({
        "Date": raw.index,
        "Close": raw["Close"][ticker].values,
        "Volume": raw["Volume"][ticker].values
    })

    df = create_features(df)
    nf_df = pd.DataFrame({
        "unique_id": ticker,
        "ds": pd.to_datetime(df["Date"]),
        "y": df["Close"],
        
        "volume": df["Volume"],
        "sma20": df["sma20"],
        "sma50": df["sma50"],
        "ema20": df["ema20"],
        "rsi": df["rsi"],
        "macd": df["macd"],
        "macd_signal": df["macd_signal"],
        "bb_low": df["bb_low"],
        "bb_high": df["bb_high"],
        "returns": df["returns"],
        "volatility": df["volatility"],
    })

    train_size = len(nf_df) - FORECAST_HORIZON

    train_df = nf_df.iloc[:train_size]
    test_df = nf_df.iloc[train_size:]

    model = PatchTST(
        h=FORECAST_HORIZON,
        input_size=252,
        patch_len=16,
        stride=8,
        hidden_size=64,
        n_heads=4,
        learning_rate=1e-3,
        max_steps=300,
        batch_size=32
    )

    nf = NeuralForecast(
        models=[model],
        freq='D'
    )

    # Train
    nf.fit(df=train_df)

    forecast = nf.predict()
    preds = forecast["PatchTST"].values

    # evaluation
    y_true = test_df["y"].values[:len(preds)]
    mae = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    dstat = directional_accuracy(y_true, preds)

    # calculation
    current_price = df["Close"].iloc[-1]
    forecast_price = preds[-1]
    volatility = df["volatility"].iloc[-1]
    expected_return = ((forecast_price - current_price) / current_price) * 100
    all_forecasts.append({
        "Ticker": ticker,
        "CurrentPrice": current_price,
        "ForecastPrice": forecast_price,
        "ExpectedReturn": expected_return,
        "MAE": mae,
        "RMSE": rmse,
        "Dstat": dstat,
        "Volatility": volatility
    })


    plt.figure(figsize=(12,6))
    plt.plot(test_df["ds"][:len(preds)], y_true, label="Actual")
    plt.plot(test_df["ds"][:len(preds)], preds, label="Forecast")

    plt.title(f"{ticker} Forecast")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.grid(True)

    plot_pdf.savefig()
    plt.close()
plot_pdf.close()

Seed set to 1
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type              | Params | Mode 
-----------------------------------------------------------
0 | loss         | MAE               | 0      | train
1 | padder_train | ConstantPad1d     | 0      | train
2 | scaler       | TemporalNorm      | 0      | train
3 | model        | PatchTST_backbone | 391 K  | train
-----------------------------------------------------------
391 K     Trainable params
3         Non-trainable params
391 K     Total params
1.565     Total estimated model params size (MB)
90        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=300` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

In [24]:
EXPORT_FORECAST_PORTOFOLIO = "./reports/all_stock_forecast.csv"
EXPORT_FORECAST_TOP5 = "./reports/top5_portofolio.csv"

In [42]:
result_df = pd.DataFrame(all_forecasts)
result_df = result_df.sort_values(
    by="ExpectedReturn",
    ascending=False
)

print("\n==============================")
print("ALL STOCK RANKING")
print("==============================")

print(result_df)

top5 = result_df.head(TOP_K).copy()
total_return = top5["ExpectedReturn"].sum()
top5["Weight"] = (top5["ExpectedReturn"]/ top5["ExpectedReturn"].sum())


TOTAL_FUNDS = 1_000_000_000_000
top5["Allocation"] = (top5["Weight"] * TOTAL_FUNDS)
top5["ProjectedProfit"] = (top5["Allocation"] * top5["ExpectedReturn"] / 100)

print("\n==============================")
print("TOP 5 PORTFOLIO")
print("==============================")

print(top5)

result_df.to_csv(EXPORT_FORECAST_PORTOFOLIO, index=False)
top5.to_csv(EXPORT_FORECAST_TOP5, index=False)

print("\nSaved:")
print(f"- {EXPORT_FORECAST_PORTOFOLIO}")
print(f"- {EXPORT_FORECAST_TOP5}")


ALL STOCK RANKING
  Ticker  CurrentPrice  ForecastPrice  ExpectedReturn        MAE       RMSE  \
1    AXP    316.470001     427.011841       34.929642  55.319509  68.122709   
3    BAC     51.333282      67.720749       31.923668   7.899868   9.502799   
5    MCO    453.250000     515.252991       13.679645  57.315361  67.034847   
0   AAPL    312.059998     308.071808       -1.278020  23.096665  26.506281   
7     CB    311.730011     296.888031       -4.761165  28.284395  32.028734   
8    KHC     23.582586      22.263964       -5.591510   0.875615   1.044371   
2     KO     79.010002      69.368721      -12.202608   8.520813   9.656082   
6    OXY     56.630001      46.927486      -17.133171   8.186199  10.009272   
9  GOOGL    380.112946     296.829712      -21.910128  60.638806  70.188472   
4    CVX    182.460007     135.555191      -25.706902  33.530707  37.737943   

      Dstat  Volatility  
1  0.487395    0.008929  
3  0.512605    0.013873  
5  0.495798    0.015463  
0  0.46

In [43]:
total_profit = top5["ProjectedProfit"].sum()

In [44]:
print("\n==============================")
print("TOTAL PROJECTED PROFIT")
print("==============================")
print(f"Total Profit: Rp {total_profit:,.0f}")


TOTAL PROJECTED PROFIT
Total Profit: Rp 328,971,830,231


In [45]:
portfolio_return = top5["ExpectedReturn"].dot(top5["Weight"])
print(f"Portfolio Return: {portfolio_return:.2f}%")

Portfolio Return: 32.90%


In [46]:
import pandas as pd

df_stock = pd.read_csv(EXPORT_FORECAST_TOP5)
tickers = df_stock["Ticker"].tolist()

In [47]:
results = []
for ticker in tickers:
    df = pd.DataFrame({
        "Date": raw.index,
        "Close": raw["Close"][ticker].values,
        "Volume": raw["Volume"][ticker].values
    })

    df = create_features(df)
    nf_df = pd.DataFrame({
        "unique_id": ticker,
        "ds": pd.to_datetime(df["Date"]),
        "y": df["Close"],
        
        "volume": df["Volume"],
        "sma20": df["sma20"],
        "sma50": df["sma50"],
        "ema20": df["ema20"],
        "rsi": df["rsi"],
        "macd": df["macd"],
        "macd_signal": df["macd_signal"],
        "bb_low": df["bb_low"],
        "bb_high": df["bb_high"],
        "returns": df["returns"],
        "volatility": df["volatility"],
    })

    model = PatchTST(
        h=FORECAST_HORIZON,
        input_size=252,
        patch_len=16,
        stride=8,
        hidden_size=64,
        n_heads=4,
        learning_rate=1e-3,
        max_steps=300,
        batch_size=32
    )
    
    nf = NeuralForecast(
        models=[model],
        freq='D'
    )
    
    # TRAIN ALL HISTORICAL DATA
    nf.fit(df=nf_df)
    
    future_forecast = nf.predict()
    forecast_series = future_forecast["PatchTST"]

    # Buy Point
    lowest_idx = forecast_series.idxmin()
    lowest_price = forecast_series.loc[lowest_idx]
    lowest_horizon = lowest_idx + 1
    
    
    future_after_buy = forecast_series.loc[lowest_idx:]
    
    # Sell
    highest_idx = future_after_buy.idxmax()
    highest_price = future_after_buy.loc[highest_idx]
    highest_horizon = highest_idx + 1

    # Expected return
    expected_return = (
        (highest_price - lowest_price)
        / lowest_price
    ) * 100
    
   
    results.append({
        "Ticker": ticker,
        "HighestPrice": highest_price,
        "HighestHorizon": f"H+{highest_horizon}",
        "LowestPrice": lowest_price,
        "LowestHorizon": f"H+{lowest_horizon}",
        "ExpectedReturn": expected_return,
    })

Seed set to 1
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type              | Params | Mode 
-----------------------------------------------------------
0 | loss         | MAE               | 0      | train
1 | padder_train | ConstantPad1d     | 0      | train
2 | scaler       | TemporalNorm      | 0      | train
3 | model        | PatchTST_backbone | 391 K  | train
-----------------------------------------------------------
391 K     Trainable params
3         Non-trainable params
391 K     Total params
1.565     Total estimated model params size (MB)
90        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=300` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

In [40]:
EXPORT_TOP5_FORECAST = "./reports/top5_forecast.csv"

In [41]:
import pandas as pd

result_df = pd.DataFrame(results)
result_df = result_df.sort_values(
    by="ExpectedReturn",
    ascending=False
)
top5 = result_df.head(5).copy()
top5["Weight"] = (top5["ExpectedReturn"] / top5["ExpectedReturn"].sum())

TOTAL_FUNDS = 1_000_000_000_000
top5["Allocation"] = (top5["Weight"] * TOTAL_FUNDS)
top5["ProjectedProfit"] = (top5["Allocation"] * top5["ExpectedReturn"]/ 100)

top5.to_csv(EXPORT_TOP5_FORECAST, index=False)

total_profit = top5["ProjectedProfit"].sum()
print(top5)

print("\n====================")
print("TOTAL PROFIT")
print("====================")

print(f"Rp {total_profit:,.0f}")

  Ticker  HighestPrice HighestHorizon  LowestPrice LowestHorizon  \
0    AXP    463.085907          H+103   321.866608           H+1   
3   AAPL    289.182281          H+120   204.812225          H+68   
1    BAC     55.714241          H+120    50.012691          H+67   
4     CB    330.422974          H+118   306.113190          H+50   
2    MCO    444.545166          H+118   415.453735         H+106   

   ExpectedReturn    Weight    Allocation  ProjectedProfit  
0       43.875103  0.393806  3.938062e+11     1.727829e+11  
3       41.193859  0.369740  3.697403e+11     1.523103e+11  
1       11.400206  0.102324  1.023239e+11     1.166513e+10  
4        7.941436  0.071279  7.127930e+10     5.660600e+09  
2        7.002327  0.062850  6.285022e+10     4.400978e+09  

TOTAL PROFIT
Rp 346,819,887,104
